# 14. 결정 API — 생성하지 않고 고른다 (OpenAI Decisions API · Jev)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/15_decisions_api.ipynb)

> 🧭 **스토리** — 규정 도우미의 뒤쪽에는 글을 쓰지 않는 일이 많다. 문의를 어느 부서로 보낼지 **고르고**, 개인정보가 섞였는지 **판정하고**, 답이 맞았는지 **채점한다**. 이런 일에 긴 답을 생성시키면 느리고 비싸다. TypeSafe의 Jev가 연 "결정 모델" 범주를 OpenAI가 Decisions API로 따라왔다. 고르는 일은 고르는 모델에게 맡긴다.
>
> 📖 **이론 연결** — 이론2 p82~87 (결정 모델 · Decisions API 3타입 · 파이프라인 안의 위치 · 구조화 출력과 비교) · ⏱ **12분** · 🖥 **CPU**
>
> GPU가 필요 없습니다. 기본 런타임(CPU)으로 실행하세요.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
Decisions API(`POST /v1/decisions`, 2026-10 공개 베타, 보도 기준)는 `openai` 3.26 이상에 들어 있습니다. 모델은 `gpt-6-luna`만 지원합니다.

| 질문 타입 | 묻는 것 | 돌려받는 것 |
|---|---|---|
| `predicate` | "이 문장이 참인가?" | `probability` (0~1) |
| `choice` | "보기 중 무엇인가?" (2~255개) | `choice`, `confidence`, 보기별 `probabilities` |
| `score` | "낮음→높음 등급 중 어디인가?" | `score`(등급 번호의 기대값, 소수 가능), `confidence`, `probabilities` |

글을 생성하지 않으므로 **출력 토큰 과금이 없습니다.** 입력 $0.10 / 1M 토큰만 냅니다.

In [ ]:
%pip -q install "openai>=3.26" pandas

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-data/genai)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-data/main/genai/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.
print("키 입력 완료 · 모델:", MODEL_API)

In [ ]:
# ⚙️ 비용 계산 — 2026-10 기준 공식 단가(USD per 1M tokens). https://developers.openai.com/api/docs/pricing
PRICES = {  # (입력, 캐시 입력, 출력)
    "gpt-6-luna":   (0.10, 0.01, 0.50),
    "gpt-5-nano":   (0.05, 0.005, 0.40),
    "gpt-5.6-luna": (0.20, 0.02, 1.20),
    "gpt-6-astra":  (10.0, 1.0, 50.0),
    "text-embedding-3-small": (0.02, 0.02, 0.0),
}
def cost_usd(model: str, in_tok: int, out_tok: int = 0, cached_tok: int = 0) -> float:
    p_in, p_cache, p_out = PRICES.get(model, (0.10, 0.01, 0.50))
    return ((in_tok - cached_tok) * p_in + cached_tok * p_cache + out_tok * p_out) / 1_000_000
def cost_krw(usd: float, rate: float = 1400.0) -> float:
    return usd * rate
PRICES["gpt-6-luna:decisions"] = (0.10, 0.0, 0.0)   # Decisions API: 입력만 과금(출력·캐시 과금 없음)

In [ ]:
# ⚙️ decide() 헬퍼 — 결정 요청 + 지연 측정. 실패하면 한국어 안내만 찍고 (None, 지연)을 돌려줍니다.
import time, json, openai
from openai import OpenAI
client = OpenAI()

def explain_error(e):
    """오류 종류와 상태 코드만 출력합니다(응답 본문은 출력하지 않음)."""
    status, code = getattr(e, "status_code", "-"), getattr(e, "code", None)
    if isinstance(e, openai.AuthenticationError):
        print(f"🔑 {type(e).__name__}({status}) — 키가 없거나 틀렸습니다. 키 셀을 다시 실행하세요. 이 셀의 API 부분은 건너뜁니다.")
    elif isinstance(e, openai.RateLimitError) and code == "project_spend_limit_exceeded":
        print(f"💸 {type(e).__name__}({status}) — 수업용 프로젝트의 지출 하드 한도에 도달했습니다. 강사에게 알려 주세요.")
    elif isinstance(e, openai.RateLimitError):
        print(f"⏳ {type(e).__name__}({status}) — 속도 제한입니다. 잠시 뒤 다시 실행하세요.")
    else:
        print(f"⚠️ {type(e).__name__}({status})")

def decide(input, questions):
    t0 = time.time()
    try:
        d = client.decisions.create(model=MODEL_API, input=input, questions=questions)
        return d, time.time() - t0
    except Exception as e:
        explain_error(e)
        return None, time.time() - t0

def by_name(d):
    """답을 질문 이름으로 묶습니다. 답은 질문 순서대로 옵니다. refusal(답 거부)이면 None."""
    return {a.name: (None if a.type == "refusal" else a) for a in d.answers}

## 1. 첫 결정 — 문의 라우팅(`choice`) 🔵
사내 문의 8건을 네 부서 중 하나로 보냅니다. 보기마다 `description`을 달면 모델이 경계를 더 잘 가립니다. 일부러 애매한 문의도 두 건 넣었습니다.

In [ ]:
INQUIRIES = [
    "다음 주 부산 출장 숙박비를 법인카드로 결제해도 되나요?",
    "연차를 반차로 나눠 쓸 수 있나요?",
    "장인어른 칠순인데 경조사 휴가가 나오나요?",
    "AWS 자격증 응시료도 교육비로 지원되나요?",
    "사내 메신저가 아침부터 접속이 안 됩니다.",
    "오늘 점심 뭐 먹을지 추천해 주세요.",
    "해외 학회 참가비는 출장비인가요, 교육비인가요?",    # 애매 1
    "출장 다녀온 다음 날 쉬고 싶은데 어떻게 하나요?",     # 애매 2
]
DEPT = {"type": "choice", "name": "department",
        "instructions": "한빛테크 인사팀으로 들어온 사내 문의를 담당 영역 하나로 분류한다.",
        "choices": [
            {"value": "출장비", "description": "국내·해외 출장의 일비·식비·숙박비·교통비·정산"},
            {"value": "휴가·경조사", "description": "연차, 경조사 휴가, 경조금"},
            {"value": "교육비", "description": "직무·어학·학위·자격증 교육비 지원"},
            {"value": "other", "description": "위 셋에 해당하지 않음(IT 장애, 잡담 등)"},
        ]}

ROUTED = []   # (문의, 고른 부서, confidence)
for q in INQUIRIES:
    d, sec = decide(q, [DEPT])
    if d is None:
        break
    a = by_name(d)["department"]
    if a is None:
        print(f"{q[:24]:<26} → (refusal)"); ROUTED.append((q, None, 0.0)); continue
    ROUTED.append((q, a.choice, a.confidence))
    top2 = sorted(a.probabilities, key=lambda p: -p.probability)[:2]
    print(f"{q[:24]:<26} → {a.choice:<6} conf {a.confidence:.2f} | "
          + ", ".join(f"{p.value} {p.probability:.2f}" for p in top2) + f" | {sec:.2f}초")

💡 **해석** — 답마다 **확률**이 함께 옵니다. 그래서 "confidence가 0.6 미만이면 사람 담당자에게 넘긴다" 같은 규칙을 코드로 쓸 수 있습니다(바로 아래 셀). 애매한 두 문의에서 1, 2위 확률이 가까운지 보세요. 생성 API에 "부서 이름만 답해"라고 시키면 답 하나만 오고 확신의 정도는 알 수 없습니다. 이 호출은 **생성 토큰이 0**이므로 출력 비용도 0입니다.

In [ ]:
HANDOFF = 0.6   # confidence가 이보다 낮으면 사람 담당자에게
for q, choice, conf in ROUTED:
    route = "👤 사람 담당자" if (choice is None or conf < HANDOFF) else f"🤖 {choice} 자동 처리"
    print(f"{conf:.2f} {route:<16} | {q}")
if not ROUTED:
    print("앞 셀의 API 호출이 실패해서 건너뜁니다.")

💡 **해석** — 몇 건이 사람에게 갔나요? 애매한 두 문의가 걸렸다면 임계값이 제 역할을 한 것입니다. 임계값을 높이면 사람의 일이 늘고 오분류가 줄어듭니다. 5절에서 정답 붙인 예시로 이 값을 정하는 법을 봅니다.

## 2. 세 타입을 한 번에 🔵
같은 입력에 질문 세 개를 **한 요청**으로 묻습니다. 공식 문서에 따르면 질문은 서로 독립적으로 답합니다. 앞 질문의 답이 뒤 질문에 영향을 주지 않습니다.

In [ ]:
ANSWERABLE = {"type": "predicate", "name": "answerable",
              "instructions": "이 문의는 한빛테크 출장·복리후생 규정집(출장비, 휴가, 경조사, 교육비)으로 답할 수 있는 질문이다."}
URGENCY = {"type": "score", "name": "urgency",
           "instructions": "이 문의의 처리 긴급도를 평가한다.",
           "levels": [{"label": "낮음", "description": "언제 답해도 됨"},
                      {"label": "보통", "description": "며칠 안에 답하면 됨"},
                      {"label": "높음", "description": "오늘 안에 처리하지 않으면 업무 차질"}]}
text = "내일 아침 9시 부산 출장인데 아직 전자결재 승인을 못 받았어요. 어떻게 하죠?"
d, sec = decide(text, [ANSWERABLE, DEPT, URGENCY])
if d:
    ans = by_name(d)
    for name, a in ans.items():
        if a is None:
            print(f"{name}: refusal — 이 질문만 답을 거부했습니다"); continue
        if a.type == "predicate":
            print(f"{name}: probability {a.probability:.2f}")
        elif a.type == "choice":
            print(f"{name}: {a.choice} (confidence {a.confidence:.2f})")
        elif a.type == "score":
            print(f"{name}: score {a.score:.2f} (confidence {a.confidence:.2f}) | "
                  + ", ".join(f"{p.value}:{p.label} {p.probability:.2f}" for p in a.probabilities))
    u = d.usage
    print(f"\n입력 {u.input_tokens} · 출력 {u.output_tokens} · 캐시 {u.input_tokens_details.cached_tokens} · {sec:.2f}초 · "
          f"${cost_usd('gpt-6-luna:decisions', u.input_tokens):.6f}")

💡 **해석** — `score`는 등급 번호(낮음 0, 보통 1, 높음 2)의 **확률 가중 평균**입니다. 그래서 1.7처럼 소수가 나옵니다. "보통과 높음 사이, 높음 쪽"이라는 뜻입니다. 정렬 기준으로 쓰기 좋습니다. `refusal`은 질문 하나 단위로 옵니다. 그래서 **답을 읽기 전에 항상 `type == "refusal"`을 확인**합니다. `by_name`이 그 일을 합니다.

## 3. 생성 API의 구조화 출력과 비교 🔵
11번에서 쓴 `responses.parse` + Pydantic으로 같은 라우팅을 하고, 지연·토큰·비용을 나란히 놓습니다.

In [ ]:
import pandas as pd
from typing import Literal
from pydantic import BaseModel

class Route(BaseModel):
    department: Literal["출장비", "휴가·경조사", "교육비", "other"]

def route_responses(q):
    t0 = time.time()
    try:
        r = client.responses.parse(model=MODEL_API, input=q, text_format=Route, reasoning={"effort": "none"},
                                   instructions="한빛테크 사내 문의를 담당 영역 하나로 분류한다.")
        return r, time.time() - t0
    except Exception as e:
        explain_error(e); return None, time.time() - t0

rows = []
for method in ["Decisions", "Responses.parse"]:
    lat = tin = tout = 0.0; ok = True
    for q in INQUIRIES:
        res, sec = decide(q, [DEPT]) if method == "Decisions" else route_responses(q)
        if res is None:
            ok = False; break
        lat += sec; tin += res.usage.input_tokens; tout += res.usage.output_tokens
    n = len(INQUIRIES)
    if ok:
        usd = cost_usd("gpt-6-luna:decisions", tin) if method == "Decisions" else cost_usd(MODEL_API, tin, tout)
        rows.append({"방법": method, "지연(초/건)": round(lat / n, 2), "입력토큰(합)": int(tin), "출력토큰(합)": int(tout),
                     "비용(USD, 8건)": round(usd, 6), "확률 제공": "예" if method == "Decisions" else "아니오"})
    else:
        rows.append({"방법": method, "지연(초/건)": "키 없음", "입력토큰(합)": "-", "출력토큰(합)": "-",
                     "비용(USD, 8건)": "-", "확률 제공": "예" if method == "Decisions" else "아니오"})
display(pd.DataFrame(rows))

💡 **해석** — 실측값은 코랩에서 수업 키로 실행하면 채워집니다. 확인할 점은 세 가지입니다.
1. **출력 토큰**: Decisions는 출력이 과금되지 않습니다. Responses는 JSON 몇 토큰이 출력 단가($0.50)로 나갑니다.
2. **지연**: OpenAI는 Decisions가 생성 방식보다 약 10배 빠르다고 주장합니다. 공식 주장이므로 표의 실측으로 확인하세요.
3. **확률**: Responses는 답 하나만 줍니다. 그래서 "애매하면 사람에게"를 구현하려면 따로 장치가 필요합니다.

## 4. RAG 답 채점기 — `score` + `predicate` 🔵
노트북 11의 로컬 RAG 답 6개를 채점합니다. 10에서는 정규식으로 "정답 키워드가 들어 있나"만 셌습니다. 이번에는 결정 모델이 정답과 비교해 **틀림 / 부분 일치 / 정확**을 매깁니다.
아래 답은 노트북 11 강사 실측 출력의 앞부분을 그대로 쓰고, 잘린 뒷부분은 같은 오류 유형으로 짧게 이어 붙였습니다.

In [ ]:
QA = {q["id"]: q for q in load_json("hanbit_qa.json")}   # 필드: id, type, q, a, keywords
SAMPLES = [   # (문항 id, 모델 답) — 노트북 11 로컬(EXAONE 1.2B) 답 발췌
    (3,  "[1] 표 2의 일비(과장 이하) 기준으로 European 지역의 daily rate는 60 USD이며, 차장은 과장 이하에 포함되므로 60달러입니다."),
    (13, "과장 직급의 일비와 숙박비 상한 합계는 표 1을 기준으로 계산합니다. 1. 일비: 30,000원 × 3일 = 90,000원 "
         "2. 숙박비: 110,000원 × 2박 = 220,000원 → 합계 300,000원입니다. [1]"),
    (15, "재택근무에 대한 구체적인 일수는 제공된 근거 자료에서 명시적으로 언급되지 않습니다. 다만, 제4조(총칙) 제3조(용어 정의)에 "
         "따르면 근무 형태는 부서장이 정하므로 주 2~3일 정도가 일반적일 것입니다."),
    (2,  "[1] 표 1의 직급 \"부장\" 행에서 숙박비 상한은 1박에 150,000원입니다."),
    (7,  "본인 결혼 시 경조금은 500,000원이며 화환은 지급됩니다. [1]"),
    (11, "출장비 정산은 복귀 후 7일 이내에 해야 합니다. [1] 제17조(출장비 정산) (1)항에 근거합니다."),
]
GRADE = {"type": "score", "name": "accuracy",
         "instructions": "모델 답이 정답과 비교해 얼마나 정확한지 평가한다. 표현 차이는 무시하고 사실(숫자·조건)만 본다.",
         "levels": [{"label": "틀림", "description": "핵심 사실이 정답과 다르거나 근거 없는 추측을 답으로 냄"},
                    {"label": "부분 일치", "description": "일부 사실만 맞거나 맞는 내용에 틀린 추측이 섞임"},
                    {"label": "정확", "description": "정답의 핵심 사실을 모두 맞게 말함"}]}
SAYS_UNKNOWN = {"type": "predicate", "name": "says_unknown",
                "instructions": "모델 답은 '규정집에 근거가 없다(모른다)'고 말하고 거기서 멈춘다."}
rows = []
for qid, ans in SAMPLES:
    item = QA[qid]
    d, _ = decide(f"질문: {item['q']}\n정답: {item['a']}\n모델 답: {ans}", [GRADE, SAYS_UNKNOWN])
    if d is None:
        break
    a = by_name(d)
    rows.append({"id": qid, "유형": item["type"], "정답": item["a"][:18],
                 "정확도 score(0~2)": None if a["accuracy"] is None else round(a["accuracy"].score, 2),
                 "모른다 P": None if a["says_unknown"] is None else round(a["says_unknown"].probability, 2),
                 "정답 키워드 포함(11번 방식)": any(k in ans for k in item["keywords"]) if item["keywords"] else "키워드 없음"})
if rows:
    display(pd.DataFrame(rows))

💡 **해석** — 마지막 열이 노트북 11의 정규식 채점입니다. 정규식은 키워드가 **있다/없다**만 압니다. Q13처럼 단가는 맞고 덧셈만 틀린 답, Q15처럼 "언급되지 않습니다"라고 해 놓고 "다만…"으로 추측을 붙인 답은 **부분 일치**인데, 정규식으로는 이 중간 등급을 줄 수 없습니다. Q15는 키워드가 없는 문항이라 정규식 채점이 아예 불가능합니다. 기대: Q3·Q13은 낮은 score, Q15는 `모른다 P`가 중간쯤, 나머지 셋은 2에 가까운 score. 다만 채점기도 모델이므로 **사람이 매긴 정답 등급 몇십 개로 먼저 검증**하고 써야 합니다.

## 5. 바꿔서 실행 🟡 — 가드레일 임계값
predicate 두 개로 입력 가드레일을 만듭니다. "개인정보가 들어 있나"와 "규정 범위 밖 질문인가"입니다. 사람이 미리 정답을 붙인 8건으로 확률과 정답을 비교합니다.
1. `THRESH = 0.5`로 실행해 오탐(FP)과 미탐(FN)을 셉니다.
2. `THRESH = 0.8`로 바꿔 다시 실행합니다. 어떤 오류가 늘고 어떤 오류가 줄었나요?

In [ ]:
THRESH = 0.5   # ← 0.8로 바꿔 보세요
PII = {"type": "predicate", "name": "pii", "instructions": "입력에 주민등록번호, 계좌번호, 전화번호 같은 개인정보가 들어 있다."}
OUT = {"type": "predicate", "name": "out_of_scope",
       "instructions": "입력은 한빛테크 출장·휴가·경조사·교육비 규정과 관계없는 질문이다."}
LABELED = [   # (문장, 개인정보 정답, 범위 밖 정답) — 사람이 붙인 정답
    ("출장비 입금 계좌가 국민 123-45-678901로 바뀌었어요.", True, False),
    ("제 주민번호 900101-1234567로 경조금 신청됐는지 확인해 주세요.", True, False),
    ("연차 문의는 010-1234-5678로 전화 주세요.", True, False),
    ("과장 국내 출장 일비가 얼마예요?", False, False),
    ("어학 교육비는 월 얼마까지 나와요?", False, False),
    ("주식 투자 종목 좀 추천해 주세요.", False, True),
    ("회사 근처 맛집 알려 주세요. 제 번호는 010-9876-5432예요.", True, True),
    ("사번 20231234인데 연차 며칠 남았어요?", False, False),   # 사번은 개인정보일까? 애매한 사례
]
fp = {"pii": 0, "out_of_scope": 0}; fn = {"pii": 0, "out_of_scope": 0}; done = True
for text, y_pii, y_out in LABELED:
    d, _ = decide(text, [PII, OUT])
    if d is None:
        done = False; break
    a = by_name(d)
    line = f"{text[:30]:<32}"
    for name, y in [("pii", y_pii), ("out_of_scope", y_out)]:
        if a[name] is None:
            line += f" | {name} refusal"; continue
        p = a[name].probability; pred = p >= THRESH
        fp[name] += pred and not y; fn[name] += (not pred) and y
        line += f" | {name} {p:.2f}{'✔' if pred == y else '✘'}"
    print(line)
if done:
    print(f"\nTHRESH={THRESH} → 오탐(FP) {fp} · 미탐(FN) {fn}")

💡 **해석** — 공식 가이드는 임계값을 감으로 정하지 말고 **정답 붙인 예시로 정하라**고 권합니다. 임계값을 올리면 차단이 줄어 오탐(FP)이 줄고 미탐(FN)이 늘어납니다. 어느 쪽이 더 비싼지가 기준입니다. 개인정보는 놓치면 사고이므로 임계값을 낮게 두어 FN을 줄이고, 오탐은 "개인정보를 지우고 다시 보내 주세요" 안내로 흡수합니다. 범위 밖 질문은 잘못 막으면 직원이 불편하므로 임계값을 높게 둡니다. 8건은 연습용입니다. 실무에서는 수백 건으로 정합니다.

## 6. 이미지도 고른다 🔵
Decisions API는 이미지도 받습니다. 단, 이미지는 **base64 data URL**로 직접 넣어야 합니다. http 주소나 업로드 파일 id는 받지 않습니다. 요청 하나에 128장까지입니다. `detail="low"`로 토큰을 아낍니다.

In [ ]:
import base64
def data_url(path):
    return "data:image/png;base64," + base64.b64encode(open(path, "rb").read()).decode()

DOC_KIND = {"type": "choice", "name": "doc_kind", "instructions": "이미지가 어떤 문서인지 고른다.",
            "choices": [{"value": "영수증"}, {"value": "차트"}, {"value": "규정 본문 페이지"}, {"value": "기타"}]}
HAS_TABLE = {"type": "predicate", "name": "money_table", "instructions": "이미지에 금액이 적힌 표가 보인다."}
for name in ["images/receipt.png", "images/chart_allowance.png", "scan/p2.png"]:
    msg = [{"role": "user", "type": "message",
            "content": [{"type": "input_text", "text": "한빛테크 직원이 올린 첨부 파일이다."},
                        {"type": "input_image", "image_url": data_url(fetch(name)), "detail": "low"}]}]
    d, sec = decide(msg, [DOC_KIND, HAS_TABLE])
    if d is None:
        break
    a = by_name(d)
    kind = "refusal" if a["doc_kind"] is None else f"{a['doc_kind'].choice}({a['doc_kind'].confidence:.2f})"
    table = "refusal" if a["money_table"] is None else f"{a['money_table'].probability:.2f}"
    print(f"{name:<28} → {kind} · 금액 표 P {table} · 입력 {d.usage.input_tokens}토큰 · {sec:.2f}초")

💡 **해석** — 기대: 영수증 → 영수증, 수당 차트 → 차트, 스캔 2쪽(표 1 국내 출장비) → 규정 본문 페이지이고 금액 표 확률이 높게 나와야 합니다. `detail="low"`면 이미지 크기와 상관없이 토큰이 적게 듭니다. 2MB짜리 스캔도 적은 토큰으로 처리됩니다. 첨부 파일 분류(영수증이면 정산 흐름, 규정 페이지면 검색 흐름)에 바로 쓸 수 있습니다.

## 7. 🟠 Jev 코드 미리보기 — 실행하지 않음
TypeSafe의 **Jev**(2026-09-15 조기 액세스)는 이 "결정 모델(System One)" 범주를 먼저 연 모델입니다. 유료 키와 대기열이 필요해서 오늘은 **코드만** 봅니다. 두 서비스 모두 모델 가중치를 공개하지 않습니다(독점 모델).

| | TypeSafe Jev | OpenAI Decisions API |
|---|---|---|
| 공개 | 2026-09-15 조기 액세스(대기열) | 2026-10 공개 베타, 보도 기준 |
| 단가(입력 1M) | $0.042 | $0.10 (출력·캐시 과금 없음) |
| 입력 | 텍스트만 | 텍스트 + 이미지 |
| 컨텍스트 | state 32k / 전체 64k | 미공개 |
| 질문 타입 | choice / score / noul | choice / score / predicate |
| 보정 주장 | "0.8이면 집계상 약 80% 맞음" | 확률·confidence 제공 |
| 모델 | 비공개 | gpt-6-luna |

엔드포인트는 TypeSafe 직접 호출 `POST https://api.typesafe.ai/v1/systemone`(모델 `jev-latest`)과 OpenRouter `POST https://openrouter.ai/api/alpha/decisions`(모델 `typesafe/jev-1.13`) 두 가지입니다. 요청 본문은 `state`(입력 글)와 `questions`(이름 → 질문 사전)로 이루어집니다. 질문은 `type`과 `instructions`를 가지며, `choice`는 `criteria`가 **{보기: 설명} 사전**, `score`는 `criteria`가 **낮음→높음 순서의 설명 배열**(최대 10단계), `noul`(≈ OpenAI `predicate`)은 `criteria`가 없습니다. 응답은 질문 이름으로 묶여 오고, `choice`/`probabilities`/`confidence`, `score`/`legend`/`probabilities`/`confidence`, `noul`(confidence 없음)과 `usage.cost`가 들어 있습니다(OpenRouter Jev 해설 기준, 2026-10). OpenAI와 가장 다른 점: Jev는 **선택지 설명을 사전으로**, OpenAI는 **`{value, description}` 목록으로** 받습니다.

In [ ]:
RUN_JEV = False   # 유료 키가 있을 때만 True. 오늘 수업에서는 실행하지 않습니다.
# OpenRouter 경유 예시 (https://openrouter.ai/blog/insights/what-is-jev/ 의 형식). TypeSafe 직접 호출은 POST https://api.typesafe.ai/v1/systemone, 모델 "jev-latest".
body = {
    "model": "typesafe/jev-1.13",
    "state": "다음 주 부산 출장 숙박비를 법인카드로 결제해도 되나요?",
    "questions": {
        "department": {"type": "choice", "instructions": "어느 담당이 처리하나?",
                       "criteria": {"출장비": "출장·숙박·일비·교통비", "휴가·경조사": "연차·경조 휴가·경조금",
                                    "교육비": "어학·자격증·교육 지원", "other": "그 외 전부"}},
        "urgency":    {"type": "score", "instructions": "얼마나 급한가?",
                       "criteria": ["다음 주 이후여도 됨", "이번 주 안에 답해야 함", "오늘 안에 답해야 함"]},   # 낮음→높음
        "answerable": {"type": "noul", "instructions": "출장·복리후생 규정집으로 답할 수 있는 질문인가?"},   # noul ≈ predicate
    },
}
if RUN_JEV:
    import requests, os
    r = requests.post("https://openrouter.ai/api/alpha/decisions", json=body, timeout=30,
                      headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
    print(r.status_code, r.json())   # answers.department.choice / .urgency.score·legend / .answerable.noul, usage.cost
else:
    print("RUN_JEV=False — Jev는 유료 키가 필요해 실행하지 않습니다. 위 body 구조만 눈으로 비교하세요.")

### 🤔 생각해보기

1) 규정 도우미에서 **라우팅은 Decisions로, 답 생성은 Responses로** 나누면 비용과 지연이 어떻게 바뀔까요?

2) 13번의 출력 가드레일(대괄호가 있나)을 `predicate`로 바꾸면 무엇이 좋아지고 무엇을 조심해야 할까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

1) 모든 문의가 먼저 Decisions를 한 번 거칩니다. 입력만 과금되고 생성 토큰이 없으므로 이 단계는 싸고 빠릅니다. 잡담·범위 밖 문의(`other`)는 여기서 끝나므로 검색·생성 비용이 0이 됩니다. 규정 문의만 RAG + Responses로 갑니다. confidence가 낮은 문의는 사람에게 보내 오답 비용을 줄입니다. 전체 지연은 결정 한 번만큼 늘지만, 생성을 건너뛰는 문의가 많을수록 평균 비용은 내려갑니다.

2) 좋은 점: "근거가 실제로 답을 뒷받침하는가"처럼 대괄호 검사로는 못 보는 **의미**를 판정할 수 있고, 확률이 나오므로 임계값을 조정할 수 있습니다. 조심할 점: 모델 호출이 하나 늘어 지연·비용이 생깁니다. 채점 모델도 틀릴 수 있으므로 정답 붙인 예시로 임계값을 정하고 주기적으로 다시 검증해야 합니다. 대괄호 검사처럼 규칙으로 확실히 잡히는 것은 규칙으로 두고, 의미 판정만 predicate에 맡기는 2단 구성이 실용적입니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 라우팅·가드레일·채점은 '쓰는 일'이 아니라 '고르는 일'이다. 결정 모델에 맡기면 출력 비용이 0이고 확률이 나온다.
- Decisions API 질문 타입은 predicate(참일 확률), choice(보기 선택), score(등급 기대값) 세 가지다.
- 답을 읽기 전에 항상 refusal부터 확인한다. 답은 질문 순서대로 온다.
- 임계값은 정답 붙인 예시로 정하고, 오탐과 미탐 중 무엇이 더 비싼지로 고른다.
- 이미지는 base64 data URL로만 넣는다. Jev는 텍스트 전용·유료라 오늘은 코드만 봤다.

> 다음 노트북: **오늘 실습 끝! 다음 차시(2~4차시)에서 네이버 CLOVA Studio로 같은 코드를 옮겨 봅니다.**